# AI Learning Assistant - RAG Prototype (LangChain Runnable)

This notebook follows the project instructions and attached system design exactly:

1. Ingest PDFs and notebooks from `data/raw`
2. Chunk text with LangChain `RecursiveCharacterTextSplitter` (500 / 100)
3. Build three Chroma indexes:
   - Courses
   - Parts (with `course` metadata)
   - Chunks (with `course`, `section`, `source_file`, `chunk_id`)
4. Query pipeline:
   - Retrieve course
   - Retrieve parts filtered by course
   - Retrieve top 20 chunks filtered by course
   - Rerank with `BAAI/bge-reranker-base` and keep top 5
   - Compress context and generate answer with `llama3.1`
5. Guardrails are enforced in the final prompt:
   - Out-of-scope questions are handled politely
   - Broad/ambiguous questions request clarification
   - Answers stay grounded in retrieved context

In [9]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

import langchain
from chromadb import PersistentClient
from chromadb.api.shared_system_client import SharedSystemClient
from chromadb.utils import embedding_functions
from langchain_core.globals import set_debug, set_verbose
from langchain_classic.memory import ConversationSummaryBufferMemory
from langchain_ollama import ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.runnables import RunnableBranch, RunnableLambda
from nbformat import read as nb_read
from pypdf import PdfReader
from sentence_transformers import CrossEncoder

langchain.debug = True
set_debug(True)
set_verbose(True)

EMBEDDING_MODEL = "nomic-embed-text"
LLM_MODEL = "llama3.1"
RERANKER_MODEL = "BAAI/bge-reranker-base"

CHUNK_SIZE_TOKENS = 500
CHUNK_OVERLAP_TOKENS = 100
TOP_K_COURSES = 3
TOP_K_PARTS = 8
TOP_K_CHUNKS = 20
TOP_K_FINAL = 5

In [32]:
def clean_course_name(file_path: Path) -> str:
    """Map file name to a normalized course name."""
    name = file_path.stem.lower().strip()
    name = re.sub(r"[_\-]+", " ", name)
    name = re.sub(r"\s+", " ", name)
    return name


def extract_pdf_text(file_path: Path) -> str:
    """Extract text from PDF and keep pages connected for cross-page chunks."""
    reader = PdfReader(str(file_path))
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(pages)


def extract_notebook_text(file_path: Path) -> str:
    """Extract all notebook cell content as plain text."""
    with file_path.open("r", encoding="utf-8") as file:
        nb = nb_read(file, as_version=4)

    blocks: list[str] = []
    for cell in nb.cells:
        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)
        if source.strip():
            blocks.append(source)
    return "\n\n".join(blocks)


def normalize_chunk_text(text: str) -> str:
    """Normalize chunk text for retrieval while preserving punctuation."""
    normalized = text.replace("\r\n", "\n").replace("\r", "\n")
    normalized = re.sub(r"(?<=\w)-\s*\n\s*(?=\w)", "", normalized)
    normalized = re.sub(r"\s*\n+\s*", " ", normalized)
    normalized = re.sub(r"\s+", " ", normalized).strip()
    return normalized.lower()


def detect_sections(text: str) -> list[tuple[str, str]]:
    """Split by numbered headings like '1.intro' or fallback to one section."""
    pattern = re.compile(r"^\s*(\d+\.[^\n]+)", re.MULTILINE)
    matches = list(pattern.finditer(text))

    if not matches:
        return [("0.general", text)]

    sections: list[tuple[str, str]] = []
    for i, match in enumerate(matches):
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        section_name = match.group(1).strip().lower()
        section_text = text[start:end].strip()
        sections.append((section_name, section_text))
    return sections


def build_chunker() -> RecursiveCharacterTextSplitter:
    """LangChain recursive chunker with 500/100 token-like sizing."""
    return RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE_TOKENS,
        chunk_overlap=CHUNK_OVERLAP_TOKENS,
        length_function=lambda txt: len(txt.split()),
        separators=["\n\n", "\n", ". ", " ", ""],
    )


def build_documents(raw_data_dir: Path) -> tuple[list[Document], list[Document], list[Document]]:
    """Create documents for course, part, and chunk collections."""
    chunker = build_chunker()

    course_docs: list[Document] = []
    part_docs: list[Document] = []
    chunk_docs: list[Document] = []

    seen_courses: set[str] = set()
    seen_parts: set[tuple[str, str]] = set()

    for file_path in raw_data_dir.rglob("*"):
        if not file_path.is_file() or file_path.suffix.lower() not in {".pdf", ".ipynb"}:
            continue

        course = clean_course_name(file_path)
        if course not in seen_courses:
            seen_courses.add(course)
            course_docs.append(Document(page_content=course, metadata={"course": course}))

        full_text = extract_pdf_text(file_path) if file_path.suffix.lower() == ".pdf" else extract_notebook_text(file_path)
        for section, section_text in detect_sections(full_text):
            part_key = (course, section)
            if part_key not in seen_parts:
                seen_parts.add(part_key)
                part_docs.append(
                    Document(page_content=section, metadata={"course": course, "section": section})
                )

            for idx, chunk_text in enumerate(chunker.split_text(section_text)):
                normalized_chunk = normalize_chunk_text(chunk_text)
                if not normalized_chunk:
                    continue

                chunk_docs.append(
                    Document(
                        page_content=normalized_chunk,
                        metadata={
                            "course": course,
                            "section": section,
                            "source_file": file_path.name,
                            "chunk_id": f"chunk::{course}::{section}::{idx}",
                        },
                    )
                )

    return course_docs, part_docs, chunk_docs

In [33]:
def locate_workspace_root() -> Path:
    """Resolve workspace root when running from backend/rag notebook folder."""
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "raw").exists():
        return cwd
    if cwd.name == "rag" and (cwd.parent.parent / "data" / "raw").exists():
        return cwd.parent.parent
    return cwd


def create_chroma_client(persist_dir: Path) -> PersistentClient:
    """Create Chroma client for a persistent local vector DB."""
    return PersistentClient(path=str(persist_dir))


def reset_collections(persist_dir: Path, collection_names: list[str]) -> None:
    """Drop prototype collections to avoid duplicate indexing."""
    SharedSystemClient.clear_system_cache()
    client = create_chroma_client(persist_dir)
    for name in collection_names:
        try:
            client.delete_collection(name)
        except ValueError:
            pass


def upsert_documents(
    collection: Any,
    documents: list[Document],
    id_prefix: str,
    batch_size: int = 64,
 ) -> None:
    """Insert notebook-built documents into a Chroma collection in batches."""
    if not documents:
        return

    for start in range(0, len(documents), batch_size):
        batch = documents[start : start + batch_size]
        ids: list[str] = []
        texts: list[str] = []
        metadatas: list[dict[str, Any]] = []

        for offset, doc in enumerate(batch):
            ids.append(f"{id_prefix}::{start + offset}")
            texts.append(doc.page_content)
            metadatas.append(doc.metadata)

        collection.add(ids=ids, documents=texts, metadatas=metadatas)


workspace_root = locate_workspace_root()
raw_data_dir = workspace_root / "data" / "raw"
persist_dir = workspace_root / "backend" / "vectordb" / "chroma_db"
persist_dir.mkdir(parents=True, exist_ok=True)

reset_collections(
    persist_dir,
    ["prototype_courses", "prototype_parts", "prototype_chunks"],
)

embedding_fn = embedding_functions.OllamaEmbeddingFunction(
    model_name=EMBEDDING_MODEL,
    url="http://localhost:11434/api/embeddings",
)
client = create_chroma_client(persist_dir)

course_collection = client.get_or_create_collection(
    name="prototype_courses",
    embedding_function=embedding_fn,
)
part_collection = client.get_or_create_collection(
    name="prototype_parts",
    embedding_function=embedding_fn,
)
chunk_collection = client.get_or_create_collection(
    name="prototype_chunks",
    embedding_function=embedding_fn,
)

course_docs, part_docs, chunk_docs = build_documents(raw_data_dir)
upsert_documents(course_collection, course_docs, "course")
upsert_documents(part_collection, part_docs, "part")
upsert_documents(chunk_collection, chunk_docs, "chunk")

reranker = CrossEncoder(RERANKER_MODEL)

print(f"Indexed courses: {len(course_docs)}")
print(f"Indexed parts:   {len(part_docs)}")
print(f"Indexed chunks:  {len(chunk_docs)}")

Indexed courses: 32
Indexed parts:   690
Indexed chunks:  712


In [34]:
CONTEXT_TOKEN_LIMIT = 5000
chat_llm = ChatOllama(model=LLM_MODEL, temperature=0.0, verbose=True)


def _parse_guard_response(raw_output: str) -> dict[str, str]:
    """Parse guard LLM output and normalize status/message."""
    default_message = "Your question is broad or ambiguous. Please narrow it to a specific topic, section, or example."
    cleaned = raw_output.strip()
    cleaned = re.sub(r"^```(?:json)?\\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\\s*```$", "", cleaned)

    try:
        payload = json.loads(cleaned)
        status = str(payload.get("status", "")).strip().lower()
        if status == "ok":
            return {"status": "ok"}
        if status in {"needs_clarification", "clarify", "ambiguous", "stop"}:
            message = str(payload.get("message", "")).strip()
            return {"status": "stop", "message": message or default_message}
    except json.JSONDecodeError:
        pass

    lowered = cleaned.lower()
    if "needs_clarification" in lowered or "clarify" in lowered or "ambiguous" in lowered:
        return {"status": "stop", "message": default_message}
    if '"status"' in lowered and '"ok"' in lowered:
        return {"status": "ok"}

    # Default open to avoid false rejects when model formatting drifts.
    return {"status": "ok"}


def guard_question(question: str) -> dict[str, Any]:
    """Use the model to decide whether clarification is required."""
    guard_prompt = f"""
You classify whether a user question is clear enough for retrieval in a data-science learning assistant.

Return JSON only with one of these exact formats:
{{"status":"ok"}}
{{"status":"needs_clarification","message":"<one short clarifying request>"}}

Decision rule:
- Choose "ok" if the question has a concrete topic and can reasonably be answered, even when short.
- Choose "needs_clarification" only when user intent is genuinely unclear or too broad to answer usefully.

Question:
{question}
""".strip()

    response = chat_llm.invoke(guard_prompt)
    decision = _parse_guard_response(str(response.content))
    if decision.get("status") == "ok":
        return {"status": "ok", "question": question}

    return {
        "status": "stop",
        "message": decision.get(
            "message",
            "Your question is broad or ambiguous. Please narrow it to a specific topic, section, or example.",
        ),
    }


def normalize_retrieval_query(question: str) -> str:
    """Normalize retrieval query without stripping punctuation."""
    normalized = question.lower().replace("\r\n", "\n").replace("\r", "\n")
    normalized = re.sub(r"\s*\n+\s*", " ", normalized)
    normalized = re.sub(r"\s+", " ", normalized).strip()
    return normalized or question.strip()


def query_collection(
    collection: Any,
    question: str,
    n_results: int,
    where: dict[str, Any] | None = None,
) -> list[Document]:
    """Query a Chroma collection and map results to LangChain Documents."""
    query_kwargs: dict[str, Any] = {
        "query_texts": [question],
        "n_results": n_results,
    }
    if where is not None:
        query_kwargs["where"] = where

    result = collection.query(**query_kwargs)
    documents = result.get("documents", [[]])[0]
    metadatas = result.get("metadatas", [[]])[0]
    return [
        Document(page_content=doc_text, metadata=metadata or {})
        for doc_text, metadata in zip(documents, metadatas)
    ]


def rerank_documents(question: str, documents: list[Document], top_k: int = TOP_K_FINAL) -> list[Document]:
    """Rerank with bge-reranker-base and keep top-k."""
    if not documents:
        return []

    pairs = [[question, doc.page_content] for doc in documents]
    scores = reranker.predict(pairs)
    ordered_indices = sorted(
        range(len(documents)),
        key=lambda idx: float(scores[idx]),
        reverse=True,
    )
    return [documents[idx] for idx in ordered_indices[:top_k]]


def approximate_tokens(text: str) -> int:
    """Token approximation using whitespace-separated units."""
    return len(text.split())


def format_context_block(doc: Document, index: int, body: str) -> str:
    """Format one context block with source metadata."""
    meta = doc.metadata
    return (
        f"[{index}] course={meta['course']} | section={meta['section']} | "
        f"source={meta['source_file']} | chunk_id={meta['chunk_id']}\\n{body}"
    )


def summarize_chunk_with_llm(question: str, doc: Document, index: int) -> str:
    """Summarize contextual ideas only while preserving formulas and algorithms."""
    prompt = f"""
You are compressing a retrieved context chunk for a RAG system.

Rules:
1. Preserve every formula, equation, and mathematical expression exactly as written.
2. Preserve every algorithm step, pseudocode, and code instruction exactly as written.
3. Summarize only explanatory/contextual ideas around formulas and algorithms.
4. Do not invent facts and do not add external knowledge.
5. Keep output concise and faithful to the chunk.

Question:
{question}

Chunk metadata:
course={doc.metadata['course']}, section={doc.metadata['section']}, source={doc.metadata['source_file']}, chunk_id={doc.metadata['chunk_id']}

Chunk text:
{doc.page_content}
""".strip()

    response = chat_llm.invoke(prompt)
    summary = response.content.strip()
    return format_context_block(doc, index, summary)


def compress_context(question: str, documents: list[Document], token_limit: int = CONTEXT_TOKEN_LIMIT) -> str:
    """Summarize chunks with LLM when context exceeds token limit."""
    full_blocks = [
        format_context_block(doc, i, doc.page_content)
        for i, doc in enumerate(documents, start=1)
    ]
    full_context = "\\n\\n".join(full_blocks)

    if approximate_tokens(full_context) <= token_limit:
        return full_context

    summarized_blocks = [
        summarize_chunk_with_llm(question, doc, i)
        for i, doc in enumerate(documents, start=1)
    ]
    return "\\n\\n".join(summarized_blocks)


def retrieve_payload(payload: dict[str, Any]) -> dict[str, Any]:
    """Course -> parts -> chunks retrieval, then rerank and compress."""
    question = payload["question"]
    retrieval_question = normalize_retrieval_query(question)

    # Anchor routing on chunk-level semantic hits to avoid course-name misrouting.
    global_chunk_hits = query_collection(
        chunk_collection,
        retrieval_question,
        TOP_K_CHUNKS,
    )
    if not global_chunk_hits:
        return {"status": "stop", "message": "I do not have enough retrieved context to answer this reliably."}

    top_global_course = str(global_chunk_hits[0].metadata.get("course", "")).strip()
    course_hits = query_collection(course_collection, retrieval_question, TOP_K_COURSES)
    selected_course = top_global_course or (course_hits[0].page_content if course_hits else "")
    if not selected_course:
        return {"status": "stop", "message": "No indexed course matched this question."}

    part_hits = query_collection(
        part_collection,
        retrieval_question,
        TOP_K_PARTS,
        where={"course": selected_course},
    )
    selected_parts = [doc.page_content for doc in part_hits]

    chunk_hits = [doc for doc in global_chunk_hits if doc.metadata.get("course") == selected_course]
    if not chunk_hits:
        chunk_hits = query_collection(
            chunk_collection,
            retrieval_question,
            TOP_K_CHUNKS,
            where={"course": selected_course},
        )

    if selected_parts:
        part_set = set(selected_parts)
        part_filtered = [doc for doc in chunk_hits if doc.metadata.get("section") in part_set]
        if part_filtered:
            chunk_hits = part_filtered

    ranked_docs = rerank_documents(question, chunk_hits, top_k=TOP_K_FINAL)
    if not ranked_docs:
        return {"status": "stop", "message": "I do not have enough retrieved context to answer this reliably."}

    context = compress_context(question, ranked_docs)
    return {
        "status": "ok",
        "question": question,
        "selected_course": selected_course,
        "selected_parts": selected_parts,
        "ranked_docs": ranked_docs,
        "context": context,
    }


def build_prompt(payload: dict[str, Any]) -> dict[str, Any]:
    """Build the final grounded prompt for llama3.1."""
    question = payload["question"]
    context = payload["context"]

    prompt = f"""
You are an AI learning assistant for data science course material.
Use only the provided context to answer.
If context is insufficient, say you do not have enough information.
If the question is unrelated to course material, politely explain that you can only help with course content.
If the question is ambiguous, ask a clarifying question first.
If formulas or algorithms are present in context, preserve them exactly in your answer.
Answer directly and naturally; do not start with meta-prefaces like "Based on the provided context".
Cite sources using [1], [2], etc.

Question:
{question}

Context:
{context}
""".strip()

    payload["prompt"] = prompt
    return payload


def generate_answer(payload: dict[str, Any]) -> dict[str, Any]:
    """Call llama3.1 through LangChain ChatOllama."""
    response = chat_llm.invoke(payload["prompt"])
    payload["answer"] = response.content.strip()
    return payload


def format_output(payload: dict[str, Any]) -> dict[str, Any]:
    """Return concise structured output for inspection."""
    sources = [
        {
            "course": doc.metadata["course"],
            "section": doc.metadata["section"],
            "source_file": doc.metadata["source_file"],
            "chunk_id": doc.metadata["chunk_id"],
        }
        for doc in payload["ranked_docs"]
    ]

    return {
        "answer": payload["answer"],
        "selected_course": payload["selected_course"],
        "selected_parts": payload["selected_parts"][:5],
        "sources": sources,
    }


stop_runnable = RunnableLambda(lambda x: {"answer": x["message"]})

rag_chain = (
    RunnableLambda(guard_question)
    | RunnableBranch(
        (lambda x: x["status"] != "ok", stop_runnable),
        RunnableLambda(retrieve_payload)
        | RunnableBranch(
            (lambda x: x["status"] != "ok", stop_runnable),
            RunnableLambda(build_prompt)
            | RunnableLambda(generate_answer)
            | RunnableLambda(format_output),
        ),
    )
)

In [35]:
question = "what is Gradient Descent?"
result = rag_chain.invoke(question)

print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "answer": "Gradient Descent is a popular optimization method for training machine learning models. It works by iteratively adjusting the model parameters in the direction that minimizes the loss function.\n\nThe key steps in Gradient Descent are:\n\n1. Initialize Parameters: Randomly initialize the model parameters.\n2. Compute the Gradient: Calculate the gradient (derivative) of the loss function with respect to the parameters.\n3. Update Parameters: Adjust the parameters by moving in the opposite direction of the gradient, scaled by the learning rate.\n\nThe formula for updating the parameters is:\n\nw_new = w_old - (lr * gradient)\n\nwhere:\n\n* w_new: The updated weight (parameter)\n* w_old: The current weight before the update\n* lr: Learning rate (the step size, also called alpha)\n* gradient: The derivative of the loss function with respect to the weight (dloss / dw)",
  "selected_course": "gradient descent algorithms",
  "selected_parts": [
    "2. gradient descent",
    "4

In [18]:
prompt_1 = "I am learning pandas. Explain how to filter rows with multiple conditions using boolean indexing."
response_1 = rag_chain.invoke(prompt_1)
print("Turn 1 prompt:", prompt_1)
print("\nTurn 1 answer:\n", response_1.get("answer", ""))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "I am learning pandas. Explain how to filter rows with multiple conditions using boolean indexing."
}
[chain/start] [chain:RunnableSequence > chain:guard_question] Entering Chain run with input:
{
  "input": "I am learning pandas. Explain how to filter rows with multiple conditions using boolean indexing."
}
[llm/start] [chain:RunnableSequence > chain:guard_question > llm:ChatOllama] Entering LLM run with input:
{
  "prompts": [
    "Human: You classify whether a user question is clear enough for retrieval in a data-science learning assistant.\n\nReturn JSON only with one of these exact formats:\n{\"status\":\"ok\"}\n{\"status\":\"needs_clarification\",\"message\":\"<one short clarifying request>\"}\n\nDecision rule:\n- Choose \"ok\" if the question has a concrete topic and can reasonably be answered, even when short.\n- Choose \"needs_clarification\" only when user intent is genuinely unclear or too bro

In [ ]:
prompt_2 = "Great. Build on your previous answer and show how to combine conditions with &, |, and ~ with correct parentheses."
response_2 = rag_chain.invoke(prompt_2)
print("Turn 2 prompt:", prompt_2)
print("\nTurn 2 answer:\n", response_2.get("answer", ""))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Great. Build on your previous answer and show how to combine conditions with &, |, and ~ with correct parentheses."
}
[chain/start] [chain:RunnableSequence > chain:guard_question] Entering Chain run with input:
{
  "input": "Great. Build on your previous answer and show how to combine conditions with &, |, and ~ with correct parentheses."
}
[chain/end] [chain:RunnableSequence > chain:guard_question] s] Exiting Chain run with output:
{
  "status": "ok",
  "question": "Great. Build on your previous answer and show how to combine conditions with &, |, and ~ with correct parentheses."
}
[chain/start] [chain:RunnableSequence > chain:RunnableBranch] Entering Chain run with input:
{
  "status": "ok",
  "question": "Great. Build on your previous answer and show how to combine conditions with &, |, and ~ with correct parentheses."
}
[chain/start] [chain:RunnableSequence > chain:RunnableBranch > chain:RunnableLam

In [ ]:
prompt_3 = "Now compare boolean indexing versus DataFrame.query() and tell me when each one is better."
response_3 = rag_chain.invoke(prompt_3)
print("Turn 3 prompt:", prompt_3)
print("\nTurn 3 answer:\n", response_3.get("answer", ""))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Now compare boolean indexing versus DataFrame.query() and tell me when each one is better."
}
[chain/start] [chain:RunnableSequence > chain:guard_question] Entering Chain run with input:
{
  "input": "Now compare boolean indexing versus DataFrame.query() and tell me when each one is better."
}
[chain/end] [chain:RunnableSequence > chain:guard_question] s] Exiting Chain run with output:
{
  "status": "ok",
  "question": "Now compare boolean indexing versus DataFrame.query() and tell me when each one is better."
}
[chain/start] [chain:RunnableSequence > chain:RunnableBranch] Entering Chain run with input:
{
  "status": "ok",
  "question": "Now compare boolean indexing versus DataFrame.query() and tell me when each one is better."
}
[chain/start] [chain:RunnableSequence > chain:RunnableBranch > chain:RunnableLambda] Entering Chain run with input:
{
  "status": "ok",
  "question": "Now compare boolean index

In [ ]:
prompt_4 = "Use the same context and give me a mini debugging checklist for when a pandas filter returns empty results."
response_4 = rag_chain.invoke(prompt_4)
print("Turn 4 prompt:", prompt_4)
print("\nTurn 4 answer:\n", response_4.get("answer", ""))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Use the same context and give me a mini debugging checklist for when a pandas filter returns empty results."
}
[chain/start] [chain:RunnableSequence > chain:guard_question] Entering Chain run with input:
{
  "input": "Use the same context and give me a mini debugging checklist for when a pandas filter returns empty results."
}
[chain/end] [chain:RunnableSequence > chain:guard_question] s] Exiting Chain run with output:
{
  "status": "ok",
  "question": "Use the same context and give me a mini debugging checklist for when a pandas filter returns empty results."
}
[chain/start] [chain:RunnableSequence > chain:RunnableBranch] Entering Chain run with input:
{
  "status": "ok",
  "question": "Use the same context and give me a mini debugging checklist for when a pandas filter returns empty results."
}
[chain/start] [chain:RunnableSequence > chain:RunnableBranch > chain:RunnableLambda] Entering Chain run with

In [ ]:
prompt_5 = "Take that checklist and convert it into a short step-by-step algorithm I can follow every time."
response_5 = rag_chain.invoke(prompt_5)
print("Turn 5 prompt:", prompt_5)
print("\nTurn 5 answer:\n", response_5.get("answer", ""))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Take that checklist and convert it into a short step-by-step algorithm I can follow every time."
}
[chain/start] [chain:RunnableSequence > chain:guard_question] Entering Chain run with input:
{
  "input": "Take that checklist and convert it into a short step-by-step algorithm I can follow every time."
}
[chain/end] [chain:RunnableSequence > chain:guard_question] s] Exiting Chain run with output:
{
  "status": "ok",
  "question": "Take that checklist and convert it into a short step-by-step algorithm I can follow every time."
}
[chain/start] [chain:RunnableSequence > chain:RunnableBranch] Entering Chain run with input:
{
  "status": "ok",
  "question": "Take that checklist and convert it into a short step-by-step algorithm I can follow every time."
}
[chain/start] [chain:RunnableSequence > chain:RunnableBranch > chain:RunnableLambda] Entering Chain run with input:
{
  "status": "ok",
  "question": "Take 

In [ ]:
prompt_6 = "Summarize our whole conversation in exactly 6 bullets and include one compact reusable code template."
response_6 = rag_chain.invoke(prompt_6)
print("Turn 6 prompt:", prompt_6)
print("\nTurn 6 answer:\n", response_6.get("answer", ""))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Summarize our whole conversation in exactly 6 bullets and include one compact reusable code template."
}
[chain/start] [chain:RunnableSequence > chain:guard_question] Entering Chain run with input:
{
  "input": "Summarize our whole conversation in exactly 6 bullets and include one compact reusable code template."
}
[chain/end] [chain:RunnableSequence > chain:guard_question] s] Exiting Chain run with output:
{
  "status": "ok",
  "question": "Summarize our whole conversation in exactly 6 bullets and include one compact reusable code template."
}
[chain/start] [chain:RunnableSequence > chain:RunnableBranch] Entering Chain run with input:
{
  "status": "ok",
  "question": "Summarize our whole conversation in exactly 6 bullets and include one compact reusable code template."
}
[chain/start] [chain:RunnableSequence > chain:RunnableBranch > chain:RunnableLambda] Entering Chain run with input:
{
  "status": "o

In [ ]:
rag_chain.invoke("What is ADAM optimization?")

{'answer': 'ADAM optimization is an algorithm that combines the advantages of momentum and RMSProp [3]. It uses both the first moment (mean) and second moment (variance) of gradients to adapt the learning rate for each parameter.\n\nThe formula for ADAM is:\n\nm_t = (beta1 * m_t-1) + (1 - beta1) * g_t\nv_t = (beta2 * v_t-1) + (1 - beta2) * (g_t)^2\nm_hat_t = m_t / (1 - beta1^t)\nv_hat_t = v_t / (1 - beta2^t)\nw_t = w_t-1 - [ (lr / (v_hat_t + eps)^0.5) * m_hat_t ]\n\nwhere:\n\n* m_t: first moment (moving average of the gradient)\n* v_t: second moment (moving average of the squared gradient)\n* m_hat_t / v_hat_t: bias-corrected versions of the moments\n* g_t: gradient at step t\n* beta1: decay rate for the first moment (usually 0.9)\n* beta2: decay rate for the second moment (usually 0.999)\n* t: the current time step (used for bias correction)\n* lr: learning rate\n* eps: epsilon (tiny constant)\n\nThe advantages of ADAM include fast convergence, but it requires significant memory due t